<a href="https://colab.research.google.com/github/CevdetSatarr/FlyRank-intern/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CevdetSatarr/FlyRank-intern/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Lane 2: Refresh / Content Opportunity Scoring.** This notebook is the synthesis of ML-04 through ML-10 (data contract, task framing, baseline, signal audit, model, validation audit, action playbook) into the shape the deployed paper follows exactly.

> Read `skills/README.md`, then load `writing-research-papers` + `deploying-static-pages` before working this notebook.

## 1. Question

*The research question and the decision it supports.*

**Question:** among content items with real search demand, which are most worth a human editor's review time first — and which of the signals commonly used to make that call (age, position, volume, external keyword estimates) actually hold up against this month's real warehouse data?

**Decision it supports:** a content team deciding where to spend limited refresh/review capacity this cycle, using a ranked, reason-coded shortlist instead of an unranked flag list.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:** FlyRank ML Internship warehouse (Hugging Face dataset `FlyRank/internship-warehouse`). **Tables:** `fact_content_daily_performance` (`month=2026-03` partition), left-joined to `dim_content` for static metadata (`content_created_date`, `search_volume`). **Window:** the full mid-panel month of March 2026 — the sealed final month (`_sample`, June 2026) was never touched, per the panel's own rule against developing label logic inside the outcome window.

**Scope after filtering:** 49,396 content items (35 clients), restricted to rows with `gsc_impressions >= 100` for the month — a real-demand floor decided from data available at scoring time, not from anything in the outcome window.

**Excluded, and why:** `gsc_clicks` and the raw click counts behind the proxy label are never used as model features (they're the label's own ingredients — see Methodology). Client and content identifiers are pseudonymous hashes throughout; no client name, URL, or raw query appears anywhere in this work.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Label (proxy, not ground truth):** `is_declining` — a within-month proxy comparing each item's clicks in the first half of March against the second half. This is directional and noisy at low volume, not a confirmed business outcome.

**Features (honest set):** `gsc_impressions` (month total), `gsc_avg_position` (month average), `content_age_days` (static, from `dim_content`). All three are knowable at scoring time and independent of the label's own construction.

**Explicitly excluded from features:** `gsc_clicks` / `second_half_clicks` — confirmed via a deliberate leakage test (ML-06) that including them pushes accuracy toward a near-perfect, meaningless number, the textbook symptom of a label-derived feature.

**Baseline:** a hand-built rule (`stale × declining × high_volume × search_volume`) from ML-07, and a Logistic Regression on the honest feature set, both compared against the same data and split as the final model — never quoted from a different run.

**Validation design:** `GroupShuffleSplit` keyed on `client_hash_id` (not a random row split) — rows from the same client never appear in both train and test, preventing the model from learning client identity instead of a generalizable pattern.

**Leakage checks:** (1) the deliberate label-derived-feature test above; (2) a full signal audit (ML-06) of every candidate feature against the proxy label with visible n per bucket; (3) a before/after re-test of the split design itself (ML-09) to check whether the grouped split's numbers could be trusted at face value.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [1]:
import pandas as pd

results = pd.DataFrame([
    {'model': 'Base rate (majority class)', 'accuracy': 0.575, 'precision': 0.575, 'recall': 1.000, 'precision_at_20': 0.575},
    {'model': 'Logistic Regression (honest features)', 'accuracy': 0.575, 'precision': 0.576, 'recall': 0.996, 'precision_at_20': 0.900},
    {'model': 'Random Forest (honest features)', 'accuracy': 0.615, 'precision': 0.608, 'recall': 0.934, 'precision_at_20': 0.900},
])
print('Grouped split by client_hash_id — 48,183 train rows / 1,213 test rows, 35 clients, 0 client overlap.')
results


Grouped split by client_hash_id — 48,183 train rows / 1,213 test rows, 35 clients, 0 client overlap.


,model,accuracy,precision,recall,precision_at_20
0,Base rate (majority class),0.575,0.575,1.000,0.575
1,Logistic Regression (honest features),0.575,0.576,0.996,0.900
2,Random Forest (honest features),0.615,0.608,0.934,0.900


**Reading it honestly:** Random Forest edges out both the base rate and Logistic Regression on plain accuracy (0.615 vs 0.575), but is **tied with Logistic Regression on Precision@20 (0.90 vs 0.90)** — the metric that actually matches how this lane gets used (a reviewer working top-down through a ranked queue). On the metric that matters most for the real use case, the added model complexity did not produce a measured advantage over the simpler linear model in this run.

**Signal audit results (ML-06), the four checks behind these features:**

| Signal | Test | Verdict |
|---|---|---|
| Position tier | vs. decline rate | **CONFIRMED** — clean monotonic rise, top_3 0.479 → deep 0.805 |
| Impressions (log-bucketed) | vs. decline rate | **MIXED** — clean drop q1→q3, reverses at q4 |
| `search_volume` (external estimate) | vs. real observed impressions | **FALSE** — Spearman r = −0.014 |
| Content age (staleness) | vs. decline rate, flag-linked | **FALSE** — flat ~0.54 across every age bucket |

**Validation-design robustness check (ML-09):** re-running the same Random Forest under a naive random row split versus the grouped-by-client split showed the *grouped* split scoring slightly *higher* on accuracy and recall — the opposite of the usual “random split inflates scores” pattern. With only 35 clients and a 1,213-row test set, this is read as evidence that a single grouped split is too small to trust the direction of any such gap, not evidence that grouping improved the model. See Limitations.

## 5. Limitations

*What this work cannot claim.*

**The most important tension in this work:** the recommendation engine's highest-priority archetype (“Declining Veteran”) uses content age ≥365 days as one of its three defining criteria — but the signal audit's flag-linked test found **no relationship at all** between content age and decline rate this month (flat ~0.54 decline rate across every age bucket, large n in every bucket). This is not a data error; it's disclosed here because it directly complicates the age-based part of the recommendation logic, and a reader relying on the recommendations should weight `Declining Veteran` accordingly — the position and volume parts of that archetype's logic hold up, the age part does not, this month.

**Other limits:**
- Single month, single mid-panel partition — not validated across seasons or a longer window.
- Only 35 clients; the grouped-split test set is small (1,213 rows) and its composition can shift the reported numbers — treat Precision@20 = 0.90 as directional, not a guaranteed rate on a new client base.
- `is_declining` is a within-month proxy, not a confirmed business outcome — a page can score as declining from a single noisy week of data.
- No causal design anywhere in this pipeline — nothing here supports “refreshing X will cause Y,” only “X looks worth reviewing first, because Y.”
- `search_volume` (the external keyword-level estimate) should not be trusted as a stand-in for real demand — it showed essentially zero relationship to actual observed impressions.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [2]:
archetype_counts = pd.Series({
    'Quiet Decline': 24853,
    'Too Young to Call': 15397,
    'Stable Performer': 9005,
    'Declining Veteran': 141,
})
print(f'{archetype_counts.sum():,} content items scored, month=2026-03')
archetype_counts


49,396 content items scored, month=2026-03


,0
Quiet Decline,24853
Too Young to Call,15397
Stable Performer,9005
Declining Veteran,141


| Archetype | Action | Reason code | n |
|---|---|---|---|
| Declining Veteran | `refresh_priority` | `STALE_HIGH_VALUE_DECLINE` | 141 |
| Quiet Decline | `monitor_only` | `LOW_VALUE_DECLINE` | 24,853 |
| Stable Performer | `protect_no_action` | `STABLE_HIGH_VALUE` | 9,005 |
| Too Young to Call | `wait_and_monitor` | `INSUFFICIENT_SIGNAL_AGE` | 15,397 |

**Never automated:** publishing or editing content from this queue, pruning/de-indexing based on `monitor_only`, or overriding legal/compliance/brand review — this is a decision-support shortlist for a human editor, not a work order. Full guardrails and monitoring/retrain triggers in ML-10 (`w07_action_playbook.ipynb`).

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Reused directly from `work/figures/`: `archetype_distribution.png` (bar chart) and `playbook_metrics.json` (the receipts — population decline rate, client count, archetype counts). The results table and signal-audit table above are the two tables the paper embeds directly.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ x] Every section above is filled — markdown thinking AND the code that backs it
- [ x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x] No client names, URLs, or private queries anywhere
- [ x] My claims use careful words: observed, measured, directional, decision-support
- [ x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## 5-minute demo outline (Week-8 showcase)

**Paper:** https://cevdetsatarr.github.io/FlyRank-intern/

**0:00–0:45 · The question.** FlyRank's active content grew roughly tenfold in six months, and its own research ranks "refresh mature pages before they decay" as the first recommended action. An editor can't review every page, so: *which pages should be reviewed first?*

**0:45–1:45 · The method.** One month of production search data (March 2026, 49,396 content items, 35 clients). Label: did a page's clicks drop from the first half of the month to the second? Three leakage-checked features: impressions, average position, content age. Random Forest vs. a majority-class baseline and Logistic Regression, on a split grouped by client so no client appears in both train and test. Plus a separate audit of every candidate signal against the data.

**1:45–2:45 · One chart.** Position tier vs. decline rate: 47.9% of top-3 pages declining, rising steadily to 80.5% in the deepest tier, with a large sample at every step.

![Position tier vs decline rate](../../docs/figures/position_tier_decline.png)

**2:45–3:45 · One honest result.** Content age, the signal behind "refresh old pages first", showed no relationship to within-month decline: about 54% in every age band. And the Random Forest tied the simpler Logistic Regression on Precision@20 (0.90 each), so the extra complexity didn't buy anything on the metric that matters for a review queue.

**3:45–4:30 · One recommendation.** Order the weekly review queue by search position and real impressions first. Treat age as context, not as the reason to act. Keep a human editor on every item: this is a shortlist, never an automated edit.

**4:30–5:00 · Limits and close.** Single month, only 35 clients, a proxy label, no causal claim. It complements FlyRank's lifecycle finding rather than contradicting it. Everything is reproducible from the repo.

## Two shareable cuts

**Social post (methodology):**

> I was given one month of real production search data from @FlyRank (≈49k content pages across 35 clients) and one question: which pages should an editor review first?
>
> The obvious answer is "the old ones." So before building anything, I audited every signal against the data. Position held up cleanly: 48% of top-3 pages were losing clicks, rising to 81% in the deepest results. Age didn't: about 54% in every age band.
>
> Method note: I validated the model on a split grouped by client, so it could never "learn" a client instead of a pattern. A Random Forest tied a simple Logistic Regression on the metric that mattered, and I reported it that way.
>
> Full paper, with the limits written in: https://cevdetsatarr.github.io/FlyRank-intern/

**Employer-facing summary (3 sentences):**

> I built a refresh-priority model that ranks which content pages an editor should review first, with a reason code for each page and a human-review step instead of automated action. It was trained and validated on one month of real production search data from FlyRank (49,396 content items across 35 clients), using a client-grouped split, a majority-class baseline, and a leakage audit of every feature. It showed that search position predicted short-term click decline clearly while content age did not, which turns "refresh old pages first" into a more specific, evidence-based review queue.